In [ ]:
import pandas as pd 
import matplotlib.pylab as plt
import os 
import importlib
import analysis_utils
import numpy as nps
import scipy
import sklearn.linear_model
from tqdm import trange
import json
from models.just_think.run_models import *
import pickle

path = "../human-data/play-exp/human-v-human/final-play/final_agg.json"

with open(path) as f:
    data = json.load(f)

In [ ]:
with open("surrenders_draws/draws_merged.pkl", "rb") as f:
    draw_outcomes = pickle.load(f)


In [ ]:
with open("surrenders_draws/2025-07-03_heuristic_search_eg.pkl", "rb") as f: 
    draw_outcomes_expert = pickle.load(f)

In [ ]:
len(draw_outcomes_expert['accept_reject'])

In [ ]:
draw_outcomes['accept_reject'].count('accept'), draw_outcomes['accept_reject'].count('reject')

In [ ]:
len(draw_outcomes_expert['accept_reject'])

In [ ]:
draw_outcomes_expert.keys()

In [ ]:
novice_fun_components_df = pd.read_csv("../model-data/local_model_readout_fun_features.csv")
game2think_fun = {}
for _, entry in novice_fun_components_df.iterrows():
    h_fun = entry['humans']
    game = entry['game_id']
    game2think_fun[game] = eval(h_fun)
    
with open("../human-data/play-exp/play_human_fun.json", "r") as f:
    game2play_fun = json.load(f)

In [ ]:
variant_pths = {'full': "surrenders_draws/draws_merged.pkl", 
           'rand': "surrenders_draws/draws_uni_random_merged.pkl", 
           'abl-connect': "surrenders_draws/draws_ablate_connect_merged.pkl", 
           'abl-block': "surrenders_draws/draws_ablate_block_merged.pkl", 
           'abl-both': "surrenders_draws/draws_ablate_connect_block_merged.pkl",
           }

variant2name = {'full': "Intuitive Gamer", 
           'expert': "Expert",
           'rand': "Random", 
           'abl-connect': "Only Defense", 
           'abl-block': "Only Offense", 
           'abl-both': "Ablate Both [only center]",
           'depth-3': "Depth 3"}


variant2df = {}
for variant, pth in variant_pths.items():
    with open(pth, "rb") as f: 
        draw_outcomes = pickle.load(f)
        if variant=='full': 
            draw_df_main = pd.DataFrame(draw_outcomes_expert).copy()
            draw_df_main['p1_pwin_heuristics'] = draw_df_main['p1_pwin_hs']
            draw_df_main['p1_plose_heuristics'] = draw_df_main['p1_plose_hs']
            draw_df_main['p1_pwin'] = draw_df_main['p1_pwin_hs'].apply(np.mean)
            draw_df_main['p1_plose'] = draw_df_main['p1_plose_hs'].apply(np.mean)
            draw_df_main['all_draw'] = 1- (draw_df_main['p1_pwin'] + draw_df_main['p1_plose'])
            print(draw_df_main['game_length_heuristics'].apply(np.mean).mean(), draw_df_main['game_length_hs'].apply(np.mean).mean())
            
            draw_df_main['game_length_heuristics'] = draw_df_main['game_length_hs']
            
            variant2df['expert'] = draw_df_main
        
    draw_df_main = pd.DataFrame(draw_outcomes)
    
    draw_df_main['p1_pwin'] = draw_df_main['p1_pwin_heuristics'].apply(np.mean)
    draw_df_main['p1_plose'] = draw_df_main['p1_plose_heuristics'].apply(np.mean)
    draw_df_main['all_draw'] = 1- (draw_df_main['p1_pwin'] + draw_df_main['p1_plose'])
    
    exp_rem_lens = []
    dec_times = []
    for _, entry in draw_df_main.iterrows():
        board = entry.board 
        dec_time = sum(cell != '_' for row in board for cell in row)
        game_lens = entry['game_length_heuristics']
        if type(game_lens) == float: 
            exp_rem_lens.append([game_lens - dec_time])
        else: 
            exp_rem_lens.append([l - dec_time for l in game_lens])
        dec_times.append(dec_time)
         
    draw_df_main['exp_rem_lens'] = exp_rem_lens
    draw_df_main['exp_rem_len_mean'] = [np.mean(l) for l in exp_rem_lens]
    draw_df_main['dec_time'] = dec_times
    
    draw_df_main['fun_think'] = [np.mean(game2think_fun[game]) for game in draw_df_main['game']]
    draw_df_main['fun_play'] = [np.mean(game2play_fun[game]) for game in draw_df_main['game']]
    
    variant2df[variant] = draw_df_main
    
# repeat for expert [PATCH, as sims were run separately]
draw_df_main = variant2df['expert']
exp_rem_lens = []
dec_times = []
for _, entry in draw_df_main.iterrows():
    board = entry.board 
    dec_time = sum(cell != '_' for row in board for cell in row)
    game_lens = entry['game_length_heuristics']
    if type(game_lens) == float: 
        exp_rem_lens.append([game_lens - dec_time])
    else: 
        exp_rem_lens.append([l - dec_time for l in game_lens])
    dec_times.append(dec_time)
draw_df_main['exp_rem_lens'] = exp_rem_lens
draw_df_main['exp_rem_len_mean_full'] = [np.mean(l) for l in exp_rem_lens]
draw_df_main['dec_time'] = dec_times
draw_df_main['fun_think'] = [np.mean(game2think_fun[game]) for game in draw_df_main['game']]
draw_df_main['fun_play'] = [np.mean(game2play_fun[game]) for game in draw_df_main['game']]
variant2df['expert'] = draw_df_main



    
draw_df_main

In [ ]:
n_samples = 10
fig, axs = plt.subplots(1+len(variant2df.keys()), n_samples, figsize=(26, 25))
fig.tight_layout(pad=3.0)
for i in range(n_samples):
    # Get the first n_samples entries from the surrender_df
    idx = np.random.randint(0, len(draw_df_main))
    entry = draw_df_main.iloc[idx]
    
    # Get the board state
    board = entry['board']
    
    analysis_utils.view_table_nb({}, board, 'X', fig=fig, ax=axs[0, i], no_dist_color=True, add_colorbar=False)
    axs[0, i].set_title("{}\n{}\n{}".format(entry['game'][:30], entry['game'][30:60], entry['game'][60:90]), fontsize=12)
    axs[0, i].axis('off')
    
    # Plot the surrender probabilities
    for j, (variant, df) in enumerate(variant2df.items()):
        # Get the surrender probabilities
        draw_probs = df.iloc[idx]
        axs[j+1, i].bar(['Win', 'Draw', 'Lose'], [draw_probs['p1_pwin'], draw_probs['all_draw'], draw_probs['p1_plose']])
        axs[j+1, i].set_ylim(0, 1)
        if j == 0:
            axs[j+1, i].set_title(f"Requested: {draw_probs['request_player']}\nAccept/reject: {draw_probs['accept_reject']}\nOutcome distribution for X\n{variant2name[variant]}", fontsize=12)
        else:
            axs[j+1, i].set_title(f"{variant2name[variant]}", fontsize=12)

In [ ]:
k = 6
np.random.seed(7)

n_bootstrap = 10

variant2dfs = {}

def compute_exp_outcome(outcomes, k=-1, agg=np.mean): 
    if k != -1:
        outcomes = np.random.choice(outcomes,k, replace=True)
    return float(agg(outcomes))

for variant, draw_df_main in variant2df.items():

    new_dfs = []

    for _ in range(n_bootstrap): 
        draw_df = draw_df_main.copy()


        # add a new column for the surrender players
        req_player_win = []
        req_player_loss = []
        all_draw = []
        receive_player_win= []
        receive_player_loss =[]
        req_player_win_expert = []
        req_player_loss_expert = []
        receive_player_win_expert = []
        receive_player_loss_expert = []
        all_draw_expert = []
        rem_lens= []

        for i, entry in draw_df.iterrows(): 
            request_player = entry['request_player']
            if request_player == 'X':                 
                outcome_idxs = np.arange(len(entry['p1_pwin_heuristics']))
                if k != -1: 
                    outcome_idxs = np.random.choice(outcome_idxs, size=k, replace=True)
                win_outcomes = [v for j, v in enumerate(entry['p1_pwin_heuristics']) if j in outcome_idxs] 
                loss_outcomes = [v for j, v in enumerate(entry['p1_plose_heuristics']) if j in outcome_idxs] 
                lens = [v for j,v in enumerate(entry['exp_rem_lens']) if j in outcome_idxs]
                rem_len = np.mean(lens)
                req_pwin = np.mean(win_outcomes) 
                req_plose = np.mean(loss_outcomes) 
                rec_pwin =req_plose # if X loses, O wins
                rec_plose = req_pwin
                pdraw = (1 - (req_pwin + req_plose))
            elif request_player == 'O':                 
                outcome_idxs = np.arange(len(entry['p1_pwin_heuristics']))
                if k != -1: 
                    outcome_idxs = np.random.choice(outcome_idxs, size=k, replace=True)
                win_outcomes = [v for j, v in enumerate(entry['p1_pwin_heuristics']) if j in outcome_idxs] 
                loss_outcomes = [v for j, v in enumerate(entry['p1_plose_heuristics']) if j in outcome_idxs]
                lens = [v for j,v in enumerate(entry['exp_rem_lens']) if j in outcome_idxs]
                rem_len = np.mean(lens)
                rec_pwin = np.mean(win_outcomes)
                rec_plose = np.mean(loss_outcomes)
                req_pwin = rec_plose
                req_plose = rec_pwin
                pdraw = (1 - (rec_pwin + rec_plose))
            if pdraw < 0: 
                print("ERROR: ", request_player, pdraw)
            req_player_win.append(req_pwin)
            req_player_loss.append(req_plose)
            receive_player_win.append(rec_pwin)
            receive_player_loss.append(rec_plose)
            all_draw.append(pdraw)
            rem_lens.append(rem_len)

        draw_df['req_player_win'] = req_player_win
        draw_df['req_player_lose'] = req_player_loss
        draw_df['received_player_win'] = receive_player_win
        draw_df['received_player_lose'] = receive_player_loss
        draw_df['all_draw'] = all_draw
        draw_df['exp_rem_len_mean'] = rem_lens
            
        new_dfs.append(draw_df)

    variant2dfs[variant] = new_dfs
    
import os
import pandas as pd

output_dir = "variant2dfs_feather"
os.makedirs(output_dir, exist_ok=True)

for variant_name, df_list in variant2dfs.items():
    variant_folder = os.path.join(output_dir, variant_name)
    os.makedirs(variant_folder, exist_ok=True)
    
    for i, df in enumerate(df_list):
        df.reset_index(drop=True).to_feather(os.path.join(variant_folder, f"bootstrap_{i}.feather"))
